# Fabric CU Analyzer  v3.2
Marco Pozzan — Regolo Farm

## How to use
**Single cell notebook** — edit the `CONFIGURATION` block at the top, then **Run Cell**.

| Flag | Default | Section |
|---|---|---|
| `RUN_KPI_SUMMARY`     | `True` | A — KPI cards |
| `RUN_THROTTLING`      | `True` | B — Throttling rolling analysis |
| `RUN_CHARTS`          | `True` | C — 4-panel charts |
| `RUN_TOP_PEAKS`       | `True` | D — Top N peaks |
| `RUN_ITEM_DETAIL`     | `True` | E — CU by item/workspace/operation |
| `RUN_DAX_TOP_CU`      | `True` | F — Top DAX by CU |
| `RUN_SIZING`          | `True` | G — SKU sizing recommendation |
| `RUN_RECOMMENDATIONS` | `True` | H — Actionable recommendations |
| `RUN_EXPORT`          | `True` | I — Export CSV/JSON/Delta |

> **v3.2:** single-cell architecture — config and engine merged, no inter-cell variable sharing issues.


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CONFIGURATION  ← edit this block, then Run Cell             ║
# ╚══════════════════════════════════════════════════════════════╝

SUBSCRIPTION_ID  = ''              # your Azure subscription GUID
CAPACITY_NAME    = 'regolofabric'  # leave blank for auto-detect

DAYS_BACK        = 7   # days of history for main CU metrics
GRANULARITY_MIN  = 30  # metric granularity in minutes (30 or 60)
TOP_PEAKS        = 10  # top N peaks to display

DAX_DAYS_BACK    = 2   # days back for DAX query analysis
DAX_TOP_N        = 20  # number of top DAX operations to show
DAX_MIN_DUR_MS   = 0   # minimum duration filter (ms), 0 = disabled

# Credentials — Option A: Key Vault (production)
USE_KEYVAULT     = False
KEYVAULT_NAME    = ''  # e.g. 'kv-regolofarm'
#   secrets: fabric-tenant-id, fabric-client-id, fabric-client-secret

# Credentials — Option B: direct (development only)
TENANT_ID        = ''
CLIENT_ID        = ''
CLIENT_SECRET    = ''

EXPORT_PATH      = ''  # e.g. 'Files/cu_analyzer/'  empty = disabled

# ── Feature flags: set False to skip a section ──────────────────────
RUN_KPI_SUMMARY     = True   # A — KPI cards
RUN_THROTTLING      = True   # B — Throttling rolling analysis
RUN_CHARTS          = True   # C — 4-panel charts
RUN_TOP_PEAKS       = True   # D — Top N peaks
RUN_ITEM_DETAIL     = True   # E — CU by item / workspace / operation
RUN_DAX_TOP_CU      = True   # F — Top DAX by CU (daily & hourly)
RUN_SIZING          = True   # G — SKU sizing recommendation
RUN_RECOMMENDATIONS = True   # H — Actionable recommendations
RUN_EXPORT          = True   # I — Export CSV / JSON / Delta

# ╔══════════════════════════════════════════════════════════════╗
# ║  ENGINE  ← do not edit below this line                       ║
# ╚══════════════════════════════════════════════════════════════╝

# ══════════════════════════════════════════════════════════════
#  CELL 2 — ENGINE   ← run this cell; never needs editing
# ══════════════════════════════════════════════════════════════
import json as _json, warnings, os
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta, timezone
import requests
import pandas as pd
from IPython.display import display, HTML
warnings.filterwarnings('ignore')

NAVY   = '#1B3A6B'
ORANGE = '#C55A11'
SKY    = '#BDD7EE'
SKU_CU = {'F2':2,'F4':4,'F8':8,'F16':16,'F32':32,'F64':64,
           'F128':128,'F256':256,'F512':512,'F1024':1024,'F2048':2048}
BP_AVG_OK=60;  BP_AVG_WARN=80
BP_P95_OK=70;  BP_P95_WARN=85
BP_PEAK_OK=80; BP_PEAK_WARN=100
NUM_COLS = ['cu_s','duration_s','operations','rejected','throttling_min','users']

# ── helpers ─────────────────────────────────────────────────────────
def get_token(scope):
    r = requests.post(
        f'https://login.microsoftonline.com/{_TENANT}/oauth2/v2.0/token',
        data={'grant_type':'client_credentials','client_id':_CLIENT,
              'client_secret':_SECRET,'scope':scope})
    r.raise_for_status(); return r.json()['access_token']

def _norm(df):
    df.columns = [c.split('[')[-1].rstrip(']') for c in df.columns]; return df

def _to_num(df, cols):
    for c in cols:
        if c in df.columns: df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0)
    return df

def kpi_color(v, ok, warn):
    return '#c0392b' if v>=warn else '#e67e22' if v>=ok else '#27ae60'

def kpi_card(title, value, subtitle, color):
    return (f"<div style='display:inline-block;min-width:150px;margin:6px;padding:14px 18px;"
            f"border:2px solid {color};border-radius:8px;text-align:center;vertical-align:top'>"
            f"<div style='font-size:11px;color:#666;font-weight:bold;text-transform:uppercase'>{title}</div>"
            f"<div style='font-size:28px;font-weight:bold;color:{color};margin:6px 0'>{value}</div>"
            f"<div style='font-size:11px;color:#888'>{subtitle}</div></div>")

def section_hdr(title, sub=''):
    s = f"<span style='color:#888;font-size:12px'>{sub}</span>" if sub else ''
    display(HTML(f"<div style='border-left:4px solid {ORANGE};padding-left:12px;margin:16px 0 8px'>"
                 f"<h3 style='color:{NAVY};margin:0'>{title}</h3>{s}</div>"))

def style_df(df, bar_col=None, bar_color=None, caption=''):
    bar_color = bar_color or SKY
    fmt = {k:v for k,v in {
        'cu_s':'{:,.0f}','cu_hours':'{:.2f}','cu_pct':'{:.1f}%','cu_per_op':'{:.1f}',
        'operations':'{:,.0f}','rejected':'{:,.0f}','throttling_min':'{:.1f}',
        'duration_s':'{:,.0f}','users':'{:,.0f}'
    }.items() if k in df.columns}
    s = df.style.format(fmt)
    if bar_col and bar_col in df.columns: s = s.bar(subset=[bar_col], color=bar_color)
    if caption: s = s.set_caption(caption)
    display(s)

def fmt_min(m): return '—' if m<=0 else f'{m:.0f} min' if m<60 else f'{m/60:.1f} h'
def rec_min(pct, win): return max(0.0,((pct-100)/100)*win) if pct>100 else 0.0

# ════════════════════════════════════════════════════════════════
#  SEC 0 — Auth & Capacity Discovery  (always runs)
# ════════════════════════════════════════════════════════════════
print('─'*60)
print('SEC 0 — Auth & Capacity Discovery')
print('─'*60)

if USE_KEYVAULT and KEYVAULT_NAME:
    print(f'  Key Vault: {KEYVAULT_NAME}')
    _TENANT = notebookutils.credentials.getSecret(KEYVAULT_NAME,'fabric-tenant-id')
    _CLIENT = notebookutils.credentials.getSecret(KEYVAULT_NAME,'fabric-client-id')
    _SECRET = notebookutils.credentials.getSecret(KEYVAULT_NAME,'fabric-client-secret')
elif TENANT_ID and CLIENT_ID and CLIENT_SECRET:
    _TENANT,_CLIENT,_SECRET = TENANT_ID,CLIENT_ID,CLIENT_SECRET
    print('  Credentials: direct')
else:
    raise ValueError('Missing credentials. Set USE_KEYVAULT=True or fill TENANT_ID/CLIENT_ID/CLIENT_SECRET.')

ARM_TOKEN = get_token('https://management.azure.com/.default')
try:
    FABRIC_TOKEN = get_token('https://api.fabric.microsoft.com/.default')
    print('  Tokens: ARM + Fabric')
except Exception:
    FABRIC_TOKEN = None
    print('  Tokens: ARM only')

r_caps = requests.get(
    f'https://management.azure.com/subscriptions/{SUBSCRIPTION_ID}'
    '/providers/Microsoft.Fabric/capacities?api-version=2023-11-01',
    headers={'Authorization': f'Bearer {ARM_TOKEN}'})
r_caps.raise_for_status()
caps = [{'id':c['id'],'name':c['name'],'location':c.get('location',''),
          'sku':c.get('sku',{}).get('name','?'),
          'state':c.get('properties',{}).get('state','?'),
          'rg':c['id'].split('/resourceGroups/')[1].split('/')[0]}
         for c in r_caps.json().get('value',[])]
if not caps: raise ValueError('No Fabric Capacity found.')

if CAPACITY_NAME:
    CAP = next((c for c in caps if c['name'].lower()==CAPACITY_NAME.lower()), None)
    if not CAP: raise ValueError(f'Capacity {CAPACITY_NAME!r} not found. Available: {[c["name"] for c in caps]}')
else:
    CAP = caps[0]; print(f'  Auto-selected: {CAP["name"]}')

CAP_NAME     = CAP['name']
CAP_ID       = CAP['id']
SKU          = str(CAP.get('sku','?'))
CU_TOT       = SKU_CU.get(SKU.upper(), 0)
FAB_CAP_GUID = CAP_ID.split('/')[-1]

_rows = ''.join(
    f"<tr><td><b>{c['name']}</b></td><td>{c.get('sku','?')}</td>"
    f"<td>{SKU_CU.get(str(c.get('sku','')).upper(),'?')}</td>"
    f"<td style='color:{'green' if c.get('state')=='Active' else 'orange'}'>{c.get('state','?')}</td>"
    f"<td>{c.get('location','')}</td><td>{c.get('rg','')}</td></tr>"
    for c in caps)
display(HTML(
    f"<h3 style='color:{ORANGE}'>Fabric Capacities</h3>"
    f"<table border='1' cellpadding='6' cellspacing='0' style='border-collapse:collapse;font-family:monospace'>"
    f"<tr style='background:{NAVY};color:white'><th>Name</th><th>SKU</th><th>CU</th>"
    f"<th>State</th><th>Region</th><th>RG</th></tr>{_rows}</table>"
    f"<br><b style='color:{ORANGE}'>Analysing:</b> {CAP_NAME} | {SKU} | {CU_TOT} CU"))
print(f'  CAP_ID:       {CAP_ID}')
print(f'  FAB_CAP_GUID: {FAB_CAP_GUID}')

# ════════════════════════════════════════════════════════════════
#  SEC 1 — Download CU metrics  (always runs)
# ════════════════════════════════════════════════════════════════
print()
print('─'*60)
print('SEC 1 — Download CU metrics')
print('─'*60)

PBI_TOKEN = get_token('https://analysis.windows.net/powerbi/api/.default')
PBI_HDRS  = {'Authorization': f'Bearer {PBI_TOKEN}', 'Content-Type': 'application/json'}

DATASET_ID = None
r_ws = requests.get('https://api.powerbi.com/v1.0/myorg/groups', headers=PBI_HDRS)
if r_ws.status_code == 200:
    for ws in r_ws.json().get('value',[]):
        r_ds = requests.get(f'https://api.powerbi.com/v1.0/myorg/groups/{ws["id"]}/datasets', headers=PBI_HDRS)
        if r_ds.status_code == 200:
            for ds in r_ds.json().get('value',[]):
                if 'fabric capacity metrics' in ds.get('name','').lower():
                    DATASET_ID = ds['id']
                    print(f"  Dataset: '{ds['name']}' in workspace '{ws['name']}'")
                    break
        if DATASET_ID: break
if not DATASET_ID:
    raise ValueError('Metrics App dataset not found. Ensure the app is installed and SP has Viewer access.')

END_DT   = datetime.now(timezone.utc)
START_DT = END_DT - timedelta(days=DAYS_BACK)
PERIOD   = f'last {DAYS_BACK} days'

# refresh token
PBI_TOKEN = get_token('https://analysis.windows.net/powerbi/api/.default')
PBI_HDRS  = {'Authorization': f'Bearer {PBI_TOKEN}', 'Content-Type': 'application/json'}

def dax(q):
    r = requests.post(
        f'https://api.powerbi.com/v1.0/myorg/datasets/{DATASET_ID}/executeQueries',
        headers=PBI_HDRS,
        json={'queries':[{'query':q}],'serializerSettings':{'includeNulls':True}})
    if r.status_code == 200:
        return r.json()['results'][0]['tables'][0].get('rows',[]), 200
    return [], r.status_code

print('  Querying CU Detail...')
rows_cu, sc = dax("""
EVALUATE SUMMARIZECOLUMNS(
    'CU Detail'[Window start time],'CU Detail'[Base capacity units],
    "cu_s",    SUM('CU Detail'[CU (s)]),
    "int_s",   SUM('CU Detail'[Interactive]),
    "bg_s",    SUM('CU Detail'[Background]),
    "int_del", AVERAGE('CU Detail'[Interactive delay %]),
    "int_rej", AVERAGE('CU Detail'[Interactive rejection %]),
    "bg_rej",  AVERAGE('CU Detail'[Background rejection %]))
""")
print(f'    status={sc}  rows={len(rows_cu)}')

print('  Querying Carry-Forward...')
rows_cf, sc2 = dax("""
EVALUATE SUMMARIZECOLUMNS(
    'Timepoint Overage Detail'[Window start time],'Timepoint Overage Detail'[Base capacity units],
    "cf_add",   SUM('Timepoint Overage Detail'[Carry forward add]),
    "cf_burn",  SUM('Timepoint Overage Detail'[Carry forward burndown]),
    "cf_cumul", SUM('Timepoint Overage Detail'[Cumulative carry forward]))
""")
print(f'    status={sc2}  rows={len(rows_cf)}')

DF = pd.DataFrame(); DF_CF = pd.DataFrame()

if rows_cu:
    dr = _norm(pd.DataFrame(rows_cu))
    dr['Window start time'] = pd.to_datetime(dr['Window start time'], utc=True, errors='coerce')
    dr = dr.dropna(subset=['Window start time']).sort_values('Window start time')
    dr = dr[(dr['Window start time']>=START_DT)&(dr['Window start time']<=END_DT)]
    for col in ['cu_s','int_s','bg_s','int_del','int_rej','bg_rej','Base capacity units']:
        if col in dr.columns: dr[col] = pd.to_numeric(dr[col], errors='coerce').fillna(0)
    base_s = (dr['Base capacity units']*30.0).replace(0,1)
    dr['avg_pct'] = (dr['cu_s']/base_s*100).round(2)
    dr = dr.set_index('Window start time').sort_index()
    dr = dr.resample(f'{GRANULARITY_MIN}min').agg(
        {'avg_pct':'mean','cu_s':'sum','int_del':'max','int_rej':'max','bg_rej':'max'}
    ).reset_index().rename(columns={'Window start time':'ts'})
    DF = pd.DataFrame({'ts':dr['ts'],'metric':'CUPercent',
                        'avg':dr['avg_pct'].round(2),'max':dr['avg_pct'].round(2),
                        'total':dr['cu_s'],'int_del_pct':dr.get('int_del',0),
                        'int_rej_pct':dr.get('int_rej',0),'bg_rej_pct':dr.get('bg_rej',0)})
    print(f"    DF: {len(DF)} rows | avg={DF['avg'].mean():.1f}% max={DF['avg'].max():.1f}%")

if rows_cf:
    DF_CF = _norm(pd.DataFrame(rows_cf))
    DF_CF['Window start time'] = pd.to_datetime(DF_CF['Window start time'], utc=True, errors='coerce')
    DF_CF.rename(columns={'Window start time':'ts'}, inplace=True)
    DF_CF = DF_CF[(DF_CF['ts']>=START_DT)&(DF_CF['ts']<=END_DT)]
    for col in ['cf_add','cf_burn','cf_cumul','Base capacity units']:
        if col in DF_CF.columns: DF_CF[col] = pd.to_numeric(DF_CF[col], errors='coerce').fillna(0)

if DF.empty:
    slots = pd.date_range(START_DT, END_DT, freq=f'{GRANULARITY_MIN}min', tz='UTC')
    DF = pd.DataFrame({'ts':slots,'metric':'CUPercent','avg':0.0,'max':0.0,'total':0.0})
    print('  Fallback empty DF')

# —— derived series (used by multiple sections) ——
cu  = DF[DF['metric']=='CUPercent'].copy()
thr = DF[DF['metric']=='ThrottlingPercentage'].copy()
avg_cu  = cu['avg'].mean();    med_cu = cu['avg'].median()
p75_cu  = cu['avg'].quantile(.75); p90_cu = cu['avg'].quantile(.90)
p95_cu  = cu['avg'].quantile(.95); p99_cu = cu['avg'].quantile(.99)
max_cu  = cu['avg'].max();     std_cu = cu['avg'].std()
cv_pct  = std_cu/avg_cu*100 if avg_cu>0 else 0
int_del_max = float(DF['int_del_pct'].max())*100 if 'int_del_pct' in DF.columns else 0.0
int_rej_max = float(DF['int_rej_pct'].max())*100 if 'int_rej_pct' in DF.columns else 0.0
bg_rej_max  = float(DF['bg_rej_pct'].max())*100  if 'bg_rej_pct'  in DF.columns else 0.0
cu_s_total  = float(DF['total'].sum()) if 'total' in DF.columns else 0.0
cu_hours    = cu_s_total/3600

if   bg_rej_max>0:   thr_label,thr_color,thr_value='BG REJECT', '#8e44ad',f'{bg_rej_max:.1f}%'
elif int_rej_max>0:  thr_label,thr_color,thr_value='INT REJECT','#c0392b',f'{int_rej_max:.1f}%'
elif int_del_max>0:  thr_label,thr_color,thr_value='INT DELAY', '#e67e22',f'{int_del_max:.1f}%'
else:                thr_label,thr_color,thr_value='CLEAN',     '#27ae60','0%'

cu['ts'] = pd.to_datetime(cu['ts'])
cu['hour']    = cu['ts'].dt.hour
cu['weekday'] = cu['ts'].dt.day_name()
hourly = cu.groupby('hour')['avg'].mean()

cu_s_ser  = cu.set_index('ts')['avg'].dropna().sort_index()
gran_min  = max(1,int((cu_s_ser.index[1]-cu_s_ser.index[0]).total_seconds()/60)) if len(cu_s_ser)>1 else GRANULARITY_MIN
w10,w60,w24h = max(1,round(10/gran_min)), max(1,round(60/gran_min)), max(1,round(1440/gran_min))
r10  = cu_s_ser.rolling(w10,  min_periods=1).mean()
r60  = cu_s_ser.rolling(w60,  min_periods=1).mean()
r24h = cu_s_ser.rolling(w24h, min_periods=1).mean()
n_delay,n_irej,n_bgrej = int((r10>100).sum()),int((r60>100).sum()),int((r24h>100).sum())
pk_delay,pk_irej,pk_bgrej = float(r10.max()),float(r60.max()),float(r24h.max())
_over  = cu_s_ser[cu_s_ser>100]-100
n_ov   = len(_over); avg_ov = float(_over.mean()) if n_ov else 0.0; max_ov = float(_over.max()) if n_ov else 0.0

print(f'  Metrics ready. Period: {START_DT:%Y-%m-%d} -> {END_DT:%Y-%m-%d}')

# ════════════════════════════════════════════════════════════════
#  SEC 2 — Detail download (item/ws/op)  — only if needed
# ════════════════════════════════════════════════════════════════
DF_DETAIL=DF_BY_HOUR=DF_ITEMS_MASTER=DF_BY_ITEM=DF_BY_WS=DF_BY_OP=pd.DataFrame()

if RUN_ITEM_DETAIL or RUN_DAX_TOP_CU:
    print()
    print('─'*60)
    print('SEC 2 — Item / Operation detail download')
    print('─'*60)

    print('  Items master...')
    rows_items, _ = dax("""
EVALUATE SUMMARIZECOLUMNS(
    'Items'[Capacity Id],'Items'[Item Id],'Items'[Item name],
    'Items'[Item kind],'Items'[Workspace Id],'Items'[Workspace name])
""")
    print(f'    rows={len(rows_items)}')

    print('  Metrics By Item Operation And Day...')
    rows_detail, sc = dax("""
EVALUATE SUMMARIZECOLUMNS(
    'Metrics By Item Operation And Day'[Capacity Id],
    'Metrics By Item Operation And Day'[Workspace Id],
    'Metrics By Item Operation And Day'[Item Id],
    'Metrics By Item Operation And Day'[Operation name],
    'Metrics By Item Operation And Day'[Date],
    "cu_s",          SUM('Metrics By Item Operation And Day'[CU (s)]),
    "duration_s",    SUM('Metrics By Item Operation And Day'[Duration (s)]),
    "operations",    SUM('Metrics By Item Operation And Day'[Operations]),
    "rejected",      SUM('Metrics By Item Operation And Day'[Rejected operations]),
    "throttling_min",SUM('Metrics By Item Operation And Day'[Throttling (min)]),
    "users",         SUM('Metrics By Item Operation And Day'[Users]))
""")
    print(f'    status={sc}  rows={len(rows_detail)}')

    print('  Metrics By Item And Hour...')
    rows_by_hour, sc2 = dax("""
EVALUATE SUMMARIZECOLUMNS(
    'Metrics By Item And Hour'[Capacity Id],
    'Metrics By Item And Hour'[Workspace Id],
    'Metrics By Item And Hour'[Item Id],
    'Metrics By Item And Hour'[Datetime],
    "cu_s",          SUM('Metrics By Item And Hour'[CU (s)]),
    "duration_s",    SUM('Metrics By Item And Hour'[Duration (s)]),
    "operations",    SUM('Metrics By Item And Hour'[Operations]),
    "rejected",      SUM('Metrics By Item And Hour'[Rejected operations]),
    "throttling_min",SUM('Metrics By Item And Hour'[Throttling (min)]),
    "users",         SUM('Metrics By Item And Hour'[Users]))
""")
    print(f'    status={sc2}  rows={len(rows_by_hour)}')

    print('  Metrics By Item And Operation...')
    rows_by_op, sc4 = dax("""
EVALUATE SUMMARIZECOLUMNS(
    'Metrics By Item And Operation'[Capacity Id],
    'Metrics By Item And Operation'[Workspace Id],
    'Metrics By Item And Operation'[Item Id],
    'Metrics By Item And Operation'[Operation name],
    "cu_s",          SUM('Metrics By Item And Operation'[CU (s)]),
    "duration_s",    SUM('Metrics By Item And Operation'[Duration (s)]),
    "operations",    SUM('Metrics By Item And Operation'[Operations]),
    "rejected",      SUM('Metrics By Item And Operation'[Rejected operations]),
    "throttling_min",SUM('Metrics By Item And Operation'[Throttling (min)]),
    "users",         SUM('Metrics By Item And Operation'[Users]))
""")
    print(f'    status={sc4}  rows={len(rows_by_op)}')

    def _enrich(df):
        if not DF_ITEMS_MASTER.empty and 'Item Id' in df.columns:
            return df.merge(DF_ITEMS_MASTER[['Item Id','Item name','Item kind','Workspace name']]
                            .drop_duplicates('Item Id'), on='Item Id', how='left')
        return df

    if rows_items:
        DF_ITEMS_MASTER = _norm(pd.DataFrame(rows_items))
        if 'Capacity Id' in DF_ITEMS_MASTER.columns:
            DF_ITEMS_MASTER = DF_ITEMS_MASTER[DF_ITEMS_MASTER['Capacity Id'].str.lower()==FAB_CAP_GUID.lower()]
        print(f'    Items on capacity: {len(DF_ITEMS_MASTER)}')

    if rows_detail:
        DF_DETAIL = _to_num(_enrich(_norm(pd.DataFrame(rows_detail))), NUM_COLS)
        if 'Date' in DF_DETAIL.columns:
            DF_DETAIL['Date'] = pd.to_datetime(DF_DETAIL['Date'], utc=True, errors='coerce')
            DF_DETAIL = DF_DETAIL[(DF_DETAIL['Date']>=START_DT)&(DF_DETAIL['Date']<=END_DT)]

    if rows_by_hour:
        DF_BY_HOUR = _to_num(_enrich(_norm(pd.DataFrame(rows_by_hour))), NUM_COLS)
        if 'Datetime' in DF_BY_HOUR.columns:
            DF_BY_HOUR['Datetime'] = pd.to_datetime(DF_BY_HOUR['Datetime'], utc=True, errors='coerce')
            DF_BY_HOUR = DF_BY_HOUR[(DF_BY_HOUR['Datetime']>=START_DT)&(DF_BY_HOUR['Datetime']<=END_DT)]

    if not DF_DETAIL.empty:
        gc = [c for c in ['Item name','Item kind','Workspace name'] if c in DF_DETAIL.columns]
        if gc:
            DF_BY_ITEM = (DF_DETAIL.groupby(gc,dropna=False)[NUM_COLS].sum().reset_index()
                          .sort_values('cu_s',ascending=False))
            DF_BY_ITEM['cu_hours']=(DF_BY_ITEM['cu_s']/3600).round(2)
            DF_BY_ITEM['cu_pct']=(DF_BY_ITEM['cu_s']/DF_BY_ITEM['cu_s'].sum()*100).round(1)
        if 'Workspace name' in DF_DETAIL.columns:
            DF_BY_WS = (DF_DETAIL.groupby('Workspace name',dropna=False)[NUM_COLS].sum().reset_index()
                        .sort_values('cu_s',ascending=False))
            DF_BY_WS['cu_hours']=(DF_BY_WS['cu_s']/3600).round(2)
            DF_BY_WS['cu_pct']=(DF_BY_WS['cu_s']/DF_BY_WS['cu_s'].sum()*100).round(1)

    if rows_by_op:
        DF_BY_OP = _to_num(_enrich(_norm(pd.DataFrame(rows_by_op))), NUM_COLS)
        DF_BY_OP = DF_BY_OP.sort_values('cu_s',ascending=False)
        DF_BY_OP['cu_hours']=(DF_BY_OP['cu_s']/3600).round(2)
        DF_BY_OP['cu_pct']=(DF_BY_OP['cu_s']/DF_BY_OP['cu_s'].sum()*100).round(1)

    print('  Detail download complete')

# ════════════════════════════════════════════════════════════════
#  A — KPI Summary
# ════════════════════════════════════════════════════════════════
if RUN_KPI_SUMMARY:
    print(); print('─'*60); print('A — KPI Summary'); print('─'*60)
    cards = (
        kpi_card('CU% Average',   f'{avg_cu:.1f}%',  f'OK < {BP_AVG_OK}%',   kpi_color(avg_cu, BP_AVG_OK,  BP_AVG_WARN))
      + kpi_card('CU% P95',       f'{p95_cu:.1f}%',  f'OK < {BP_P95_OK}%',   kpi_color(p95_cu, BP_P95_OK,  BP_P95_WARN))
      + kpi_card('CU% P99',       f'{p99_cu:.1f}%',  '99th percentile',       kpi_color(p99_cu, BP_P95_OK,  BP_P95_WARN))
      + kpi_card('Peak',          f'{max_cu:.1f}%',  'absolute max',          kpi_color(max_cu, BP_PEAK_OK, BP_PEAK_WARN))
      + kpi_card('Throttling',    thr_value,          thr_label,               thr_color)
      + kpi_card('CU\u00b7hours', f'{cu_hours:.1f}', 'total CU\u00b7s / 3600', NAVY)
      + kpi_card('CV Volatility', f'{cv_pct:.1f}%',  'Coeff. Variation',      kpi_color(cv_pct,50,80))
    )
    display(HTML(
        f"<div style='border-left:4px solid {ORANGE};padding-left:12px;margin-bottom:12px'>"
        f"<h3 style='color:{NAVY};margin:0'>KPI Summary — {CAP_NAME} ({SKU} \u00b7 {CU_TOT} CU)</h3>"
        f"<span style='color:#888;font-size:12px'>{PERIOD}</span></div><div>{cards}</div>"))

# ════════════════════════════════════════════════════════════════
#  B — Throttling Analysis
# ════════════════════════════════════════════════════════════════
if RUN_THROTTLING:
    print(); print('─'*60); print('B — Throttling Analysis'); print('─'*60)

    def _badge(n, lbl, col):
        bg = col if n>0 else '#27ae60'; txt = f'{n} intervals' if n>0 else 'OK'
        return (f"<span style='background:{bg};color:white;padding:3px 10px;"
                f"border-radius:12px;font-size:12px;font-weight:bold'>{lbl}: {txt}</span>")

    def _stg(label, win, eff, n, pk, rc, col):
        c = col if n>0 else '#27ae60'
        st = f"<b style='color:{c}'>{n} intervals</b>" if n>0 else "<span style='color:#27ae60'>OK</span>"
        return (f"<tr><td><b style='color:{c}'>{label}</b></td>"
                f"<td style='text-align:center'>{win}</td><td>{eff}</td>"
                f"<td style='text-align:center'>{st}</td>"
                f"<td style='text-align:center'><b style='color:{c}'>{pk:.1f}%</b></td>"
                f"<td style='text-align:center'>{fmt_min(rc)}</td></tr>")

    badges = (_badge(n_delay,'DELAY','#e67e22')+'  '+
              _badge(n_irej,'INT REJ','#c0392b')+'  '+
              _badge(n_bgrej,'BG REJ','#8e44ad'))

    tbl = (
        f"<table border='1' cellpadding='8' cellspacing='0'"
        f"style='border-collapse:collapse;font-family:monospace;width:100%'>"
        f"<tr style='background:{NAVY};color:white'><th>Stage</th><th>Window</th><th>Effect</th>"
        f"<th>Occurrences</th><th>Peak %</th><th>Est. recovery</th></tr>"
        f"<tr style='background:#f9f9f9'><td><b>0 \u2013 Overage protection</b></td>"
        f"<td style='text-align:center'>&lt; 10 min</td><td>No user effects</td>"
        f"<td colspan='3' style='text-align:center;color:#888'>N/A</td></tr>"
        + _stg('1 Interactive DELAY','10 min','+20s on interactive queries',n_delay,pk_delay,rec_min(pk_delay,10),'#e67e22')
        + _stg('2 Interactive REJECTION','60 min','Reports / queries rejected',n_irej,pk_irej,rec_min(pk_irej,60),'#c0392b')
        + _stg('3 Background REJECTION','24 hours','ALL jobs rejected',n_bgrej,pk_bgrej,rec_min(pk_bgrej,1440),'#8e44ad')
        + '</table>')

    cf_col = '#c0392b' if n_ov>10 else '#e67e22' if n_ov>0 else '#27ae60'
    cf_html = (f"<div style='margin-top:16px;padding:12px;border-left:4px solid {cf_col};background:#fafafa'>"
               f"<b>Carry-Forward</b><br>Intervals &gt;100%: "
               f"<b style='color:{cf_col}'>{n_ov}</b> | "
               f"Avg: <b>{avg_ov:.1f}%</b> | Peak: <b>{max_ov:.1f}%</b></div>")

    ev_list, in_ev, ev_start, ev_vals = [], False, None, []
    for ts_v, v in r10.items():
        if v>100 and not in_ev:  in_ev=True; ev_start=ts_v; ev_vals=[v]
        elif v>100 and in_ev:    ev_vals.append(v)
        elif v<=100 and in_ev:
            in_ev=False
            ev_list.append({'Start':str(ev_start)[:19],'End':str(ts_v)[:19],
                            'Duration (min)':len(ev_vals)*gran_min,'Peak %':round(float(max(ev_vals)),1)})
            ev_vals=[]
    if in_ev and ev_vals:
        ev_list.append({'Start':str(ev_start)[:19],'End':'ongoing',
                        'Duration (min)':len(ev_vals)*gran_min,'Peak %':round(float(max(ev_vals)),1)})
    ev_html = ''
    if ev_list:
        ev_html = f'<br><b>Throttling events ({len(ev_list)} episodes):</b>'
        ev_html += pd.DataFrame(ev_list).to_html(index=False, border=0, justify='center')

    display(HTML(
        f"<div style='border-left:4px solid {ORANGE};padding-left:12px;margin-bottom:12px'>"
        f"<h3 style='color:{NAVY};margin:0'>Throttling Analysis \u2014 {CAP_NAME}</h3></div>"
        f"<div style='margin-bottom:10px'>{badges}</div>{tbl}{cf_html}{ev_html}"))

# ════════════════════════════════════════════════════════════════
#  C — Charts
# ════════════════════════════════════════════════════════════════
if RUN_CHARTS:
    print(); print('─'*60); print('C — Charts'); print('─'*60)

    def _prow(lbl, val, ok_t, wn_t, soglia):
        c = '#c0392b' if val>=wn_t else '#e67e22' if val>=ok_t else '#27ae60'
        st= 'CRIT' if val>=wn_t else 'WARN' if val>=ok_t else 'OK'
        return (f"<tr><td><b>{lbl}</b></td>"
                f"<td style='text-align:right'><b style='color:{c}'>{val:.1f}%</b></td>"
                f"<td style='color:#888;font-size:12px'>{soglia}</td><td>{st}</td></tr>")

    ptbl = (
        f"<table border='1' cellpadding='7' cellspacing='0' style='border-collapse:collapse;font-family:monospace'>"
        f"<tr style='background:{NAVY};color:white'><th>Statistic</th><th>CU%</th><th>MS Threshold</th><th>Status</th></tr>"
        + _prow('Average', avg_cu, BP_AVG_OK, BP_AVG_WARN, f'OK < {BP_AVG_OK}%')
        + _prow('Median',  med_cu, BP_AVG_OK, BP_AVG_WARN, '\u2014')
        + _prow('P75',     p75_cu, 65, 80, '\u2014')
        + _prow('P90',     p90_cu, 70, 85, '\u2014')
        + _prow('P95',     p95_cu, BP_P95_OK, BP_P95_WARN, f'OK < {BP_P95_OK}%')
        + _prow('P99',     p99_cu, BP_P95_OK, BP_P95_WARN, '\u2014')
        + _prow('Peak',    max_cu, BP_PEAK_OK, BP_PEAK_WARN, f'OK < {BP_PEAK_OK}%')
        + f"<tr><td><b>Std Dev</b></td><td style='text-align:right'>{std_cu:.1f}%</td>"
          f"<td colspan='2' style='color:#888'>\u2014</td></tr>"
        + f"<tr><td><b>CV (volatility)</b></td><td style='text-align:right'>{cv_pct:.1f}%</td>"
          f"<td style='color:#888'>OK < 50%</td><td>{'WARN' if cv_pct>=50 else 'OK'}</td></tr></table>")
    display(HTML(f"<h3 style='color:{ORANGE}'>CU% Statistics</h3>{ptbl}"))

    fig, axes = plt.subplots(2, 2, figsize=(18, 10))
    fig.suptitle(f'Fabric CU Analyzer \u2014 {CAP_NAME} ({SKU}) \u2014 {PERIOD}',
                 fontsize=13, fontweight='bold', color=NAVY)

    ax1 = axes[0,0]
    ax1.fill_between(cu['ts'], cu['avg'], alpha=0.3, color=SKY)
    ax1.plot(cu['ts'], cu['avg'], color=NAVY, linewidth=0.8, label='CU% avg')
    ax1.axhline(100, color='red',  linestyle='--', linewidth=1, label='100% burst limit')
    ax1.axhline(80,  color=ORANGE, linestyle=':',  linewidth=1, label='80% warning')
    ax1.axhline(avg_cu, color='green', linestyle=':', linewidth=1, label=f'mean {avg_cu:.1f}%')
    ax1.set_title('CU% over time', fontweight='bold'); ax1.set_ylabel('CU%')
    ax1.legend(fontsize=8); ax1.grid(alpha=0.3)

    ax2 = axes[0,1]
    n = len(cu)
    bkts = {'idle\n<20%':(cu['avg']<20).sum()/n*100,
             'normal\n20-60%':((cu['avg']>=20)&(cu['avg']<60)).sum()/n*100,
             'elevated\n60-80%':((cu['avg']>=60)&(cu['avg']<80)).sum()/n*100,
             'warning\n80-100%':((cu['avg']>=80)&(cu['avg']<100)).sum()/n*100,
             'overload\n>100%':(cu['avg']>=100).sum()/n*100}
    bars = ax2.bar(bkts.keys(), bkts.values(),
                   color=[SKY,'#27ae60',ORANGE,'#e67e22','#c0392b'], edgecolor='white', linewidth=0.5)
    for bar, val in zip(bars, bkts.values()):
        ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3, f'{val:.1f}%',
                 ha='center', va='bottom', fontsize=9)
    ax2.set_title('Time distribution', fontweight='bold'); ax2.set_ylabel('% of time'); ax2.grid(axis='y',alpha=0.3)

    ax3 = axes[1,0]
    ax3.bar(hourly.index, hourly.values,
            color=['#c0392b' if v>=100 else '#e67e22' if v>=80 else NAVY if v>=60 else SKY for v in hourly.values])
    ax3.axhline(80,  color=ORANGE, linestyle=':',  linewidth=1, label='80% warning')
    ax3.axhline(100, color='red',  linestyle='--', linewidth=1, label='100% limit')
    ax3.set_title('Hourly profile', fontweight='bold'); ax3.set_xlabel('Hour'); ax3.set_ylabel('avg CU%')
    ax3.set_xticks(range(0,24)); ax3.legend(fontsize=8); ax3.grid(axis='y',alpha=0.3)

    ax4 = axes[1,1]
    DAYS=['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
    DAYS_S=['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
    wm = cu.groupby('weekday')['avg'].mean().reindex(DAYS)
    wx = cu.groupby('weekday')['avg'].max().reindex(DAYS)
    xd = range(len(DAYS))
    ax4.bar([x-0.2 for x in xd], wm.fillna(0).values, 0.4, label='mean', color=NAVY,   alpha=0.8)
    ax4.bar([x+0.2 for x in xd], wx.fillna(0).values, 0.4, label='peak', color=ORANGE, alpha=0.8)
    ax4.axhline(80,  color='orange', linestyle=':',  linewidth=1)
    ax4.axhline(100, color='red',    linestyle='--', linewidth=1)
    ax4.set_title('Weekly pattern', fontweight='bold')
    ax4.set_xticks(list(xd)); ax4.set_xticklabels(DAYS_S)
    ax4.set_ylabel('CU%'); ax4.legend(fontsize=9); ax4.grid(axis='y',alpha=0.3)

    plt.tight_layout()
    plt.savefig('/tmp/cu_charts.png', dpi=140, bbox_inches='tight')
    plt.show()
    print('  Charts rendered')

# ════════════════════════════════════════════════════════════════
#  D — Top Peaks
# ════════════════════════════════════════════════════════════════
if RUN_TOP_PEAKS:
    print(); print('─'*60); print('D — Top Peaks & Rolling Throttling'); print('─'*60)
    tp = cu.nlargest(TOP_PEAKS,'avg')[['ts','avg','max']].copy()
    tp['ts'] = tp['ts'].astype(str).str[:19]
    tp.columns = ['Timestamp','CU% avg','CU% max']
    tp = tp.round(2).reset_index(drop=True); tp.index += 1
    display(HTML(f"<h3 style='color:{ORANGE}'>Top {TOP_PEAKS} absolute peaks</h3>"))
    display(tp)

    fig2, axes2 = plt.subplots(3,1, figsize=(18,10), sharex=True)
    fig2.suptitle(f'Throttling rolling windows \u2014 {CAP_NAME}', fontsize=12, fontweight='bold', color=NAVY)
    for ax, data, label, col in [
        (axes2[0], r10,  'Window 10 min (Interactive DELAY)',     '#e67e22'),
        (axes2[1], r60,  'Window 60 min (Interactive REJECTION)', '#c0392b'),
        (axes2[2], r24h, 'Window 24 h   (Background REJECTION)',  '#8e44ad'),
    ]:
        ax.fill_between(r10.index, data, where=(data>100), alpha=0.3, color=col, label='active')
        ax.plot(r10.index, data, color=NAVY, linewidth=0.8)
        ax.axhline(100, color=col, linestyle='--', linewidth=1.2, label='100% threshold')
        ax.set_ylabel('CU% rolling'); ax.set_title(label, fontsize=10, fontweight='bold')
        ax.legend(fontsize=8, loc='upper right'); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('/tmp/throttling_chart.png', dpi=140, bbox_inches='tight')
    plt.show()

# ════════════════════════════════════════════════════════════════
#  E — Item / Workspace / Operation detail
# ════════════════════════════════════════════════════════════════
if RUN_ITEM_DETAIL:
    print(); print('─'*60); print('E — Item / Workspace / Operation detail'); print('─'*60)
    section_hdr(f'CU Consumption Detail \u2014 {CAP_NAME} \u2014 {PERIOD}')

    if not DF_BY_ITEM.empty:
        display(HTML(f"<b style='color:{NAVY}'>Top consumers by item</b>"))
        sc_cols = [c for c in ['Item name','Item kind','Workspace name','cu_s','cu_hours','cu_pct',
                                'operations','rejected','throttling_min','users'] if c in DF_BY_ITEM.columns]
        style_df(DF_BY_ITEM[sc_cols].head(30).reset_index(drop=True), 'cu_s', SKY, 'CU\u00b7s per item (full period)')

    if not DF_BY_WS.empty:
        display(HTML(f"<br><b style='color:{NAVY}'>Top consumers by workspace</b>"))
        sc_cols = [c for c in ['Workspace name','cu_s','cu_hours','cu_pct',
                                'operations','rejected','throttling_min'] if c in DF_BY_WS.columns]
        style_df(DF_BY_WS[sc_cols].head(20).reset_index(drop=True), 'cu_s', '#FAD7C0', 'CU\u00b7s per workspace')

    if not DF_BY_OP.empty:
        display(HTML(f"<br><b style='color:{NAVY}'>Top consumers by item + operation</b>"))
        sc_cols = [c for c in ['Item name','Item kind','Operation name','Workspace name',
                                'cu_s','cu_hours','cu_pct','operations','rejected','throttling_min'] if c in DF_BY_OP.columns]
        style_df(DF_BY_OP[sc_cols].head(30).reset_index(drop=True), 'cu_s', '#D5E8D4', 'CU\u00b7s per item+operation')

# ════════════════════════════════════════════════════════════════
#  F — Top DAX operations by CU
# ════════════════════════════════════════════════════════════════
if RUN_DAX_TOP_CU:
    print(); print('─'*60); print('F — Top DAX operations by CU'); print('─'*60)
    DAX_END_DT   = datetime.now(timezone.utc)
    DAX_START_DT = DAX_END_DT - timedelta(days=DAX_DAYS_BACK)
    _DAX_KW      = ['dataset','query','refresh','explore','interactive','dax']

    def _is_dax(op): return any(k in str(op).lower() for k in _DAX_KW) if pd.notna(op) else False

    DF_DAX_DAY = pd.DataFrame()
    if not DF_DETAIL.empty and 'Date' in DF_DETAIL.columns:
        _dd = DF_DETAIL[(DF_DETAIL['Date']>=DAX_START_DT)&(DF_DETAIL['Date']<=DAX_END_DT)].copy()
        if 'Operation name' in _dd.columns: _dd = _dd[_dd['Operation name'].apply(_is_dax)]
        if DAX_MIN_DUR_MS>0 and 'duration_s' in _dd.columns: _dd = _dd[_dd['duration_s']*1000>=DAX_MIN_DUR_MS]
        DF_DAX_DAY = _dd

    DF_DAX_HOUR = pd.DataFrame()
    if not DF_BY_HOUR.empty and 'Datetime' in DF_BY_HOUR.columns:
        DF_DAX_HOUR = DF_BY_HOUR[(DF_BY_HOUR['Datetime']>=DAX_START_DT)&(DF_BY_HOUR['Datetime']<=DAX_END_DT)].copy()

    print(f'  DAX day rows={len(DF_DAX_DAY)}  hour rows={len(DF_DAX_HOUR)}')

    # F-1  Total
    section_hdr(
        f'Top {DAX_TOP_N} DAX Operations \u2014 Total CU ({DAX_START_DT:%Y-%m-%d} \u2192 {DAX_END_DT:%Y-%m-%d})',
        f'{DAX_DAYS_BACK}-day window \u00b7 Item + Operation')
    if not DF_DAX_DAY.empty:
        gc = [c for c in ['Item name','Item kind','Workspace name','Operation name'] if c in DF_DAX_DAY.columns]
        if gc:
            df_t = (DF_DAX_DAY.groupby(gc,dropna=False)[NUM_COLS].sum().reset_index()
                    .sort_values('cu_s',ascending=False).head(DAX_TOP_N))
            df_t['cu_hours']  = (df_t['cu_s']/3600).round(2)
            df_t['cu_pct']    = (df_t['cu_s']/df_t['cu_s'].sum()*100).round(1)
            df_t['cu_per_op'] = (df_t['cu_s']/df_t['operations'].replace(0,1)).round(1)
            df_t = df_t.reset_index(drop=True); df_t.index += 1
            sc = [c for c in ['Item name','Item kind','Workspace name','Operation name',
                               'cu_s','cu_hours','cu_pct','operations','rejected','cu_per_op','duration_s','throttling_min'] if c in df_t.columns]
            style_df(df_t[sc], 'cu_s', SKY, f'Top {DAX_TOP_N} DAX by CU\u00b7s ({DAX_DAYS_BACK}-day total)')
    else:
        display(HTML("<p style='color:#888'>No DAX data for the selected window.</p>"))

    # F-2  By day
    section_hdr(f'Top {DAX_TOP_N} DAX Operations \u2014 By Day', 'Top N per calendar day')
    if not DF_DAX_DAY.empty and 'Date' in DF_DAX_DAY.columns:
        DF_DAX_DAY['Day'] = DF_DAX_DAY['Date'].dt.strftime('%Y-%m-%d')
        gc2 = [c for c in ['Day','Item name','Item kind','Workspace name','Operation name'] if c in DF_DAX_DAY.columns]
        if gc2:
            df_bd = DF_DAX_DAY.groupby(gc2,dropna=False)[NUM_COLS].sum().reset_index()
            df_bd['cu_hours']  = (df_bd['cu_s']/3600).round(2)
            df_bd['cu_pct']    = df_bd.groupby('Day')['cu_s'].transform(lambda x: (x/x.sum()*100).round(1) if x.sum()>0 else 0)
            df_bd['cu_per_op'] = (df_bd['cu_s']/df_bd['operations'].replace(0,1)).round(1)
            df_bd = (df_bd.sort_values(['Day','cu_s'],ascending=[True,False])
                     .groupby('Day',group_keys=False).head(DAX_TOP_N).reset_index(drop=True))
            df_bd.index += 1
            sc2 = [c for c in ['Day','Item name','Item kind','Workspace name','Operation name',
                                'cu_s','cu_hours','cu_pct','operations','rejected','cu_per_op','duration_s','throttling_min'] if c in df_bd.columns]
            style_df(df_bd[sc2], 'cu_s', '#FAD7C0', f'Top {DAX_TOP_N} DAX per day')
    else:
        display(HTML("<p style='color:#888'>No daily DAX data.</p>"))

    # F-3  By hour
    section_hdr(f'Top {DAX_TOP_N} DAX Consumers \u2014 By Hour', 'Highest single-hour CU slot (item aggregated)')
    if not DF_DAX_HOUR.empty and 'Datetime' in DF_DAX_HOUR.columns:
        DF_DAX_HOUR['Hour'] = DF_DAX_HOUR['Datetime'].dt.strftime('%Y-%m-%d %H:00')
        gc3 = [c for c in ['Hour','Item name','Item kind','Workspace name'] if c in DF_DAX_HOUR.columns]
        if gc3:
            df_bh = DF_DAX_HOUR.groupby(gc3,dropna=False)[NUM_COLS].sum().reset_index()
            df_bh['cu_hours']  = (df_bh['cu_s']/3600).round(2)
            df_bh['cu_pct']    = df_bh.groupby('Hour')['cu_s'].transform(lambda x: (x/x.sum()*100).round(1) if x.sum()>0 else 0)
            df_bh['cu_per_op'] = (df_bh['cu_s']/df_bh['operations'].replace(0,1)).round(1)
            df_bh = df_bh.sort_values('cu_s',ascending=False).head(DAX_TOP_N).reset_index(drop=True)
            df_bh.index += 1
            sc3 = [c for c in ['Hour','Item name','Item kind','Workspace name',
                                'cu_s','cu_hours','cu_pct','operations','rejected','cu_per_op','duration_s','throttling_min'] if c in df_bh.columns]
            style_df(df_bh[sc3], 'cu_s', '#D5E8D4', f'Top {DAX_TOP_N} items by CU\u00b7s in a single hour')
    else:
        display(HTML("<p style='color:#888'>No hourly data available.</p>"))

# ════════════════════════════════════════════════════════════════
#  G — Sizing Recommendation
# ════════════════════════════════════════════════════════════════
if RUN_SIZING:
    print(); print('─'*60); print('G \u2014 Sizing Recommendation'); print('─'*60)
    def best_sku(cu_needed):
        for nm,cu_v in sorted(SKU_CU.items(), key=lambda x:x[1]):
            if cu_v>=cu_needed: return f'{nm} ({cu_v} CU)'
        return 'F2048+ (custom)'

    if CU_TOT>0:
        avg_cu_abs = avg_cu/100*CU_TOT; p95_cu_abs = p95_cu/100*CU_TOT
        rec_lo = p95_cu_abs*1.25;       rec_hi = p95_cu_abs*1.50
        is_undersized = p95_cu>80;      is_oversized = avg_cu_abs<CU_TOT*0.30
        if is_undersized: vc,verdict='#c0392b','UNDERDIMENSIONED \u2014 upgrade recommended'
        elif is_oversized:vc,verdict='#27ae60','Probably oversized \u2014 consider downgrade'
        else:             vc,verdict='#27ae60','Sizing appropriate for current load'
        display(HTML(
            f"<div style='border-left:4px solid {ORANGE};padding-left:12px;margin-bottom:12px'>"
            f"<h3 style='color:{NAVY};margin:0'>Sizing & SKU Recommendation</h3>"
            f"<span style='color:#888;font-size:12px'>P95 + 25-50% buffer (MS Capacity Planning Guide)</span></div>"
            f"<table border='1' cellpadding='8' cellspacing='0' style='border-collapse:collapse;font-family:monospace'>"
            f"<tr style='background:{NAVY};color:white'><th>Item</th><th>Value</th></tr>"
            f"<tr><td>Current SKU</td><td><b>{SKU}</b> ({CU_TOT} CU)</td></tr>"
            f"<tr><td>CU used (avg)</td><td>{avg_cu_abs:.1f} CUs ({avg_cu:.1f}%)</td></tr>"
            f"<tr><td>CU used (P95)</td><td>{p95_cu_abs:.1f} CU ({p95_cu:.1f}%)</td></tr>"
            f"<tr><td>Recommended +25%</td><td><b>{best_sku(rec_lo)}</b> ({p95_cu_abs:.0f}\u00d71.25={rec_lo:.0f} CU)</td></tr>"
            f"<tr><td>Recommended +50%</td><td><b>{best_sku(rec_hi)}</b> ({p95_cu_abs:.0f}\u00d71.50={rec_hi:.0f} CU)</td></tr>"
            f"<tr><td colspan='2' style='text-align:center;font-size:14px;color:{vc}'><b>{verdict}</b></td></tr>"
            f"</table>"))
    else:
        print('  Unknown SKU \u2014 sizing unavailable')

# ════════════════════════════════════════════════════════════════
#  H — Actionable Recommendations
# ════════════════════════════════════════════════════════════════
if RUN_RECOMMENDATIONS:
    print(); print('─'*60); print('H \u2014 Actionable Recommendations'); print('─'*60)
    recs = []
    def rec(sev,cat,msg): recs.append((sev,cat,msg))

    if n_bgrej: rec('crit','Throttling BG Rejection',
                    f'{n_bgrej} BG Rejection intervals. ALL jobs rejected. Pause+resume capacity or upgrade SKU.')
    if n_irej:  rec('crit','Throttling Int. Rejection',
                    f'{n_irej} Interactive Rejection intervals. Enable Surge Protection or scale SKU.')
    if n_delay and not n_irej:
                rec('warn','Throttling Int. Delay',
                    f'{n_delay} intervals with +20s delay. Est. recovery: {fmt_min(rec_min(pk_delay,10))}.')
    if not n_bgrej and not n_irej and not n_delay:
                rec('ok','Throttling','No throttling in the analysed period.')

    if n_ov>10: rec('warn','Carry-Forward',f'{n_ov} intervals >100%. Distribute heavy jobs or enable Surge Protection.')
    elif n_ov>0:rec('ok','Carry-Forward',f'Limited carry-forward ({n_ov} intervals). Monitor.')
    else:       rec('ok','Carry-Forward','No carry-forward.')

    if avg_cu>=BP_AVG_WARN: rec('crit','Avg Utilisation',f'avg {avg_cu:.1f}% \u2265 {BP_AVG_WARN}%. Capacity saturated. Upgrade.')
    elif avg_cu>=BP_AVG_OK: rec('warn','Avg Utilisation',f'avg {avg_cu:.1f}% (OK<{BP_AVG_OK}%). Optimise heavy DAX/Spark.')
    else:                   rec('ok','Avg Utilisation', f'avg {avg_cu:.1f}% \u2014 good margin.')

    if p95_cu>=BP_P95_WARN: rec('crit','P95',f'P95={p95_cu:.1f}% \u2265 {BP_P95_WARN}%. Recurrent throttling risk.')
    elif p95_cu>=BP_P95_OK: rec('warn','P95',f'P95={p95_cu:.1f}% (OK<{BP_P95_OK}%). Optimise spike-causing workloads.')
    else:                   rec('ok','P95', f'P95={p95_cu:.1f}% \u2014 adequate buffer.')

    if max_cu>100:  rec('warn','Peak',f'Peak {max_cu:.1f}% > 100%. Repeated spikes \u2192 carry-forward risk.')
    elif max_cu>80: rec('warn','Peak',f'Peak {max_cu:.1f}% near 100%. Reduced burst margin.')
    else:           rec('ok','Peak', f'Peak {max_cu:.1f}% \u2014 burst margin available.')

    if cv_pct>80:   rec('warn','Volatility',f'CV={cv_pct:.1f}% \u2014 very irregular. Consider job queuing.')
    elif cv_pct>50: rec('warn','Volatility',f'CV={cv_pct:.1f}% \u2014 moderately irregular. Monitor peaks.')
    else:           rec('ok','Volatility', f'CV={cv_pct:.1f}% \u2014 uniform load.')

    if CU_TOT>0:
        if is_undersized: rec('crit','Sizing',f'P95={p95_cu:.1f}%>80% \u2192 undersized. Recommended: {best_sku(rec_lo)} \u2013 {best_sku(rec_hi)}.')
        elif is_oversized:rec('ok','Sizing',  f'Avg {avg_cu_abs:.0f}/{CU_TOT} CU ({avg_cu:.1f}%). Possibly oversized.')
        else:             rec('ok','Sizing',  f'Sizing appropriate. P95={p95_cu_abs:.0f} CU on {CU_TOT} available.')

    ph=int(hourly.idxmax()); ph_cu=float(hourly.max())
    if ph_cu>70: rec('warn','Hourly Pattern',
                     f'Peak at {ph:02d}:00 (avg {ph_cu:.1f}%). Move batches out of this window or enable Surge Protection.')

    _sc = {'ok':('','#27ae60','#f0fff4'),'warn':('','#e67e22','#fffbf0'),'crit':('','#c0392b','#fff5f5')}
    _rh = ''.join(
        f"<tr style='background:{bg}'>"
        f"<td style='text-align:center;font-size:18px'>{icon}</td>"
        f"<td style='color:{col};font-weight:bold;white-space:nowrap'>{cat}</td>"
        f"<td>{msg}</td></tr>"
        for sev,cat,msg in recs for icon,col,bg in [_sc.get(sev,('\u2022','#333','white'))])
    display(HTML(
        f"<div style='border-left:4px solid {ORANGE};padding-left:12px;margin-bottom:12px'>"
        f"<h3 style='color:{NAVY};margin:0'>Actionable Recommendations</h3></div>"
        f"<table border='0' cellpadding='10' cellspacing='2' style='width:100%;font-family:sans-serif'>{_rh}</table>"))

# ════════════════════════════════════════════════════════════════
#  I — Export
# ════════════════════════════════════════════════════════════════
if RUN_EXPORT:
    if not EXPORT_PATH:
        print('  Export skipped (EXPORT_PATH is empty)')
    else:
        print(); print('─'*60); print('I \u2014 Export to Lakehouse'); print('─'*60)
        ts_str = datetime.now().strftime('%Y%m%d_%H%M')
        base   = f'/lakehouse/default/{EXPORT_PATH}'
        os.makedirs(base, exist_ok=True)
        for df_sv, name in [
            (DF,         f'cu_metrics_{CAP_NAME}_{ts_str}.csv'),
            (DF_DETAIL,  f'cu_detail_by_item_op_day_{CAP_NAME}_{ts_str}.csv'),
            (DF_BY_HOUR, f'cu_detail_by_item_hour_{CAP_NAME}_{ts_str}.csv'),
            (DF_BY_ITEM, f'cu_top_items_{CAP_NAME}_{ts_str}.csv'),
            (DF_BY_WS,   f'cu_top_workspaces_{CAP_NAME}_{ts_str}.csv'),
            (DF_BY_OP,   f'cu_top_operations_{CAP_NAME}_{ts_str}.csv'),
        ]:
            if not df_sv.empty:
                p = f'{base}{name}'; df_sv.to_csv(p, index=False); print(f'  Saved: {p}')

        report = {
            'generated_at': datetime.now(timezone.utc).isoformat(),
            'capacity':     {'name':CAP_NAME,'sku':SKU,'cu':CU_TOT},
            'period':       {'days':DAYS_BACK,'granularity_min':GRANULARITY_MIN,
                             'start':str(START_DT)[:19],'end':str(END_DT)[:19]},
            'kpis':         {'avg':round(avg_cu,2),'p95':round(p95_cu,2),
                             'p99':round(p99_cu,2),'max':round(max_cu,2),'cv':round(cv_pct,2)},
            'throttling':   {'n_delay':n_delay,'n_irej':n_irej,'n_bgrej':n_bgrej,
                             'pk_delay':round(pk_delay,1),'pk_irej':round(pk_irej,1),'pk_bgrej':round(pk_bgrej,1)},
            'carry_forward':{'n_intervals':n_ov,'avg_overage':round(avg_ov,1),'max_overage':round(max_ov,1)},
            'recommendations': [{'sev':s,'cat':c,'msg':m} for s,c,m in (recs if RUN_RECOMMENDATIONS else [])],
        }
        jp = f'{base}cu_report_{CAP_NAME}_{ts_str}.json'
        with open(jp,'w',encoding='utf-8') as f: _json.dump(report,f,ensure_ascii=False,indent=2,default=str)
        print(f'  JSON report: {jp}')

        try:
            spark.createDataFrame(DF).write.format('delta').mode('append') \
                .option('mergeSchema','true').save('/lakehouse/default/Tables/cu_metrics_history')
            print("  Delta table 'cu_metrics_history' updated")
        except Exception as e:
            print(f'  Delta table not written: {e}')

# ════════════════════════════════════════════════════════════════
#  DONE
# ════════════════════════════════════════════════════════════════
print()
print('='*60)
print('  Fabric CU Analyzer v3.0 \u2014 COMPLETED')
print(f'  Capacity : {CAP_NAME} ({SKU} \u00b7 {CU_TOT} CU)')
print(f'  Period   : {START_DT:%Y-%m-%d} -> {END_DT:%Y-%m-%d} ({DAYS_BACK} days)')
print(f'  CU rows  : {len(DF)} | avg {avg_cu:.1f}% | P95 {p95_cu:.1f}% | peak {max_cu:.1f}%')
print('='*60)
